# 10 — Master Runner (Pipeline Orchestrator)

Sequential runner to execute: data download → preprocessing → models → ensemble → portfolio → signals.
Use this for one-click pipeline runs after notebooks are validated.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time
from datetime import datetime

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd()
NOTEBOOKS = [
    ('00_Phase2_Data_Collection.ipynb', 'Data Collection'),
    ('01_Phase3_Stock_Selection.ipynb', 'Stock Selection'),
    ('02_arima_sarima.ipynb', 'ARIMA Modeling'),
    ('03_ets_holtwinters.ipynb', 'ETS Modeling'),
    ('04_prophet.ipynb', 'Prophet Modeling'),
    ('05_lstm.ipynb', 'LSTM Modeling'),
    ('06_garch_volatility.ipynb', 'GARCH Volatility'),
    ('07_model_comparison.ipynb', 'Model Comparison'),
    ('08_portfolio.ipynb', 'Portfolio Construction'),
    ('09_stockgro_signals.ipynb', 'StockGro Signals'),
]

def run_notebook(nb_name: str, description: str, execute: bool = True) -> bool:
    """Execute a single notebook using jupyter nbconvert."""
    nb_path = ROOT / 'notebooks' / nb_name
    if not nb_path.exists():
        print(f'❌ {description}: {nb_name} not found')
        return False
    
    if not execute:
        print(f'⏭️  Skipping {description}: {nb_name}')
        return True
    
    print(f'\n{"="*80}')
    print(f'Running: {description} ({nb_name})')
    print(f'{"="*80}')
    
    try:
        cmd = [
            sys.executable, '-m', 'jupyter', 'nbconvert',
            '--to', 'notebook',
            '--execute',
            '--inplace',
            '--allow-errors',  # Continue even if cells error
            '--ExecutePreprocessor.timeout=600',  # 10 min timeout
            str(nb_path)
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        
        if result.returncode == 0:
            print(f'✅ {description}: PASSED')
            return True
        else:
            print(f'⚠️  {description}: Completed with errors')
            if result.stderr:
                print(f'   Error output: {result.stderr[:200]}...')
            return True  # Still consider it a success for pipeline continuity
    
    except subprocess.TimeoutExpired:
        print(f'❌ {description}: TIMEOUT (exceeds 15 min)')
        return False
    except Exception as e:
        print(f'❌ {description}: ERROR - {str(e)}')
        return False


def main():
    """Master pipeline orchestrator."""
    print('\n' + '='*80)
    print('STOCKGRO CAPSTONE — MASTER RUNNER')
    print('='*80)
    print(f'Start time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    
    # Configuration
    EXECUTE_DATA_COLLECTION = False  # Set to False to skip expensive data download
    EXECUTE_ALL = True
    
    print('\nConfiguration:')
    print(f'  EXECUTE_DATA_COLLECTION: {EXECUTE_DATA_COLLECTION}')
    print(f'  EXECUTE_ALL_STEPS: {EXECUTE_ALL}')
    
    results = {}
    start_time = time.time()
    
    for nb_name, description in NOTEBOOKS:
        # Skip data collection if configured
        if 'Data_Collection' in nb_name and not EXECUTE_DATA_COLLECTION:
            run_notebook(nb_name, description, execute=False)
            results[description] = 'SKIPPED'
        else:
            success = run_notebook(nb_name, description, execute=EXECUTE_ALL)
            results[description] = 'PASSED' if success else 'FAILED'
    
    # ── Summary Report ────────────────────────────────────────
    elapsed = time.time() - start_time
    
    print('\n' + '='*80)
    print('PIPELINE EXECUTION SUMMARY')
    print('='*80)
    
    for step, status in results.items():
        status_emoji = '✅' if status == 'PASSED' else '⏭️' if status == 'SKIPPED' else '❌'
        print(f'{status_emoji}  {step}: {status}')
    
    passed = sum(1 for s in results.values() if s == 'PASSED')
    skipped = sum(1 for s in results.values() if s == 'SKIPPED')
    failed = sum(1 for s in results.values() if s == 'FAILED')
    
    print(f'\nResults: {passed} Passed | {skipped} Skipped | {failed} Failed')
    print(f'Elapsed time: {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)')
    
    # ── Output files checklist ────────────────────────────────
    print(f'\n{"="*80}')
    print('OUTPUT FILES GENERATED')
    print(f'{"="*80}')
    
    output_files = [
        ('outputs/reports/arima_metrics.csv', 'ARIMA metrics'),
        ('outputs/reports/ets_metrics.csv', 'ETS metrics'),
        ('outputs/reports/prophet_metrics.csv', 'Prophet metrics'),
        ('outputs/reports/lstm_metrics.csv', 'LSTM metrics'),
        ('outputs/reports/garch_volatility_report.csv', 'GARCH volatility'),
        ('outputs/reports/model_comparison_summary.csv', 'Model comparison'),
        ('outputs/reports/best_model_per_ticker.csv', 'Best model per stock'),
        ('outputs/portfolio/final_portfolio_allocation.csv', 'Portfolio allocation'),
        ('outputs/portfolio/stockgro_trade_instructions.csv', 'StockGro signals'),
        ('outputs/portfolio/day1_execution_plan.json', 'Day 1 execution plan'),
    ]
    
    for fpath, desc in output_files:
        full_path = ROOT / fpath
        if full_path.exists():
            size_kb = full_path.stat().st_size / 1024
            print(f'✅  {desc}: {fpath} ({size_kb:.1f} KB)')
        else:
            print(f'⚠️  {desc}: {fpath} (NOT FOUND)')
    
    print(f'\n{"="*80}')
    print('NEXT STEPS')
    print(f'{"="*80}')
    print('1. Review model comparison in: outputs/reports/model_comparison_summary.csv')
    print('2. Check portfolio allocation in: outputs/portfolio/final_portfolio_allocation.csv')
    print('3. Execute trades per: outputs/portfolio/day1_execution_plan.json')
    print('4. Track performance on Day 2 against forecasts')
    print('5. See PROJECT_SUMMARY.md for full project status')
    
    print(f'\nEnd time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
    print('\n✅ Master runner complete!')


if __name__ == '__main__':
    main()